In [48]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

In [49]:
data = pd.read_csv("/content/customer.csv")

In [50]:
data.sample(5)

,age,gender,review,education,purchased
11,74,Male,Good,UG,Yes
42,30,Female,Good,PG,Yes
0,30,Female,Average,School,No
21,32,Male,Average,PG,No
9,74,Male,Good,UG,Yes


In [51]:
X_train, X_test, y_train, y_test = train_test_split(data.drop("purchased",axis=1), data["purchased"], test_size = 0.2, random_state = 23)

In [52]:
X_train["review"].value_counts()

,count
review,
Good,15
Poor,15
Average,10


In [53]:
X_train["education"].value_counts()

,count
education,
PG,16
UG,12
School,12


# **Ordinal Categories**
Categories follow a particular order
Eg: Ratings, Education, etc.

In [54]:
from sklearn.preprocessing import OrdinalEncoder

encoder = OrdinalEncoder(categories = [["Poor","Average","Good"], ["School", "PG", "UG"]])

X_train_ordinal = X_train[["review","education"]]
X_test_ordinal = X_test[["review","education"]]

X_train_encoded = encoder.fit_transform(X_train_ordinal)
X_test_encoded = encoder.transform(X_test_ordinal)

X_train_encoded = pd.DataFrame(X_train_encoded, columns=["review","education"], index=X_train.index)
X_test_encoded = pd.DataFrame(X_test_encoded, columns=["review","education"], index=X_test.index)

X_train.drop(["review","education"], axis=1, inplace=True)
X_test.drop(["review","education"], axis=1, inplace=True)

X_train["review"] = X_train_encoded["review"]
X_train["education"] = X_train_encoded["education"]

X_test["review"] = X_test_encoded["review"]
X_test["education"] = X_test_encoded["education"]

In [55]:
X_train["review"].value_counts()

,count
review,
2.0,15
0.0,15
1.0,10


In [56]:
X_train["education"].value_counts()

,count
education,
1.0,16
2.0,12
0.0,12


In [57]:
X_train.sample(5)

,age,gender,review,education
3,72,Female,2.0,1.0
7,60,Female,0.0,0.0
0,30,Female,1.0,0.0
11,74,Male,2.0,2.0
9,74,Male,2.0,2.0


In [58]:
print(data.iloc[34])
print("\n")
print(X_train[X_train["age"] == 86])

age               86
gender          Male
review       Average
education     School
purchased         No
Name: 34, dtype: object


    age gender  review  education
34   86   Male     1.0        0.0


In [59]:
print(data.iloc[29])
print("\n")
print(X_train[X_train["age"] == 83])

age               83
gender        Female
review       Average
education         UG
purchased        Yes
Name: 29, dtype: object


    age  gender  review  education
29   83  Female     1.0        2.0


# **Label Encoding**
Used to encode target values

In [61]:
from sklearn.preprocessing import LabelEncoder

encoder = LabelEncoder()

y_train_encoded = encoder.fit_transform(y_train)
y_test_encoded = encoder.transform(y_test)

y_train_encoded = pd.DataFrame(y_train_encoded)

print(y_train.head(5))
print("\n")
print(y_train_encoded.head(5))

29    Yes
10    Yes
44     No
3      No
22    Yes
Name: purchased, dtype: object


   0
0  1
1  1
2  0
3  0
4  1


# **Nominal Encoding**
Used to encode non ordered categorical values.

Eg: Gender, isPlaced? etc.

Done using OneHotEncoder

In [66]:
print(X_train["gender"].value_counts())

gender
Female    23
Male      17
Name: count, dtype: int64


1. OHE using Pandas

In [72]:
pd.get_dummies(data, columns=["gender"],dtype=int,drop_first=True)

,age,review,education,purchased,gender_Male
0,30,Average,School,No,0
1,68,Poor,UG,No,0
2,70,Good,PG,No,0
3,72,Good,PG,No,0
4,16,Average,UG,No,0
5,31,Average,School,Yes,0
6,18,Good,School,No,1
7,60,Poor,School,Yes,0
8,65,Average,UG,No,0
9,74,Good,UG,Yes,1


2. OHE using Scikit-Learn

In [75]:
from sklearn.preprocessing import OneHotEncoder

In [114]:
X_train, X_test, y_train, y_test = train_test_split(data.drop("purchased",axis=1), data["purchased"], test_size = 0.2, random_state = 23)

In [115]:
X_train.sample(5)

,age,gender,review,education
43,27,Male,Poor,PG
22,18,Female,Poor,PG
21,32,Male,Average,PG
16,59,Male,Poor,UG
1,68,Female,Poor,UG


In [116]:
# creating encoder
ohe = OneHotEncoder(drop="first", sparse_output=False, dtype="int")

# encoding gender column
X_train_encoded = ohe.fit_transform(X_train[["gender"]])
X_test_encoded = ohe.transform(X_test[["gender"]])

# joining new encoded gender column with original data
X_train_final = np.hstack((np.array(X_train.drop("gender",axis=1)), X_train_encoded))
X_train_final = pd.DataFrame(X_train_final, columns = ["age","review","education","gender"])

X_test_final = np.hstack((np.array(X_test.drop("gender",axis=1)), X_test_encoded))
X_test_final = pd.DataFrame(X_test_final, columns = ["age","review","education","gender"])

In [117]:
print(X_train_final.tail(5))
print("\n")
print(X_train.tail(5))

   age review education gender
35  38   Good        PG      0
36  74   Good        UG      1
37  39   Good    School      1
38  45   Good    School      0
39  97   Poor        PG      1


    age  gender review education
47   38  Female   Good        PG
9    74    Male   Good        UG
40   39    Male   Good    School
38   45  Female   Good    School
19   97    Male   Poor        PG


In [118]:
print(X_test_final.tail(5))
print("\n")
print(X_test.tail(5))

  age   review education gender
5  89     Good        PG      0
6  57  Average    School      0
7  65  Average        UG      0
8  34     Good        UG      0
9  48     Poor    School      1


    age  gender   review education
33   89  Female     Good        PG
20   57  Female  Average    School
8    65  Female  Average        UG
36   34  Female     Good        UG
28   48    Male     Poor    School


3. OHE in Large Categories Types

In [132]:
data = pd.read_csv("/content/cars.csv")
data.sample(5)

,brand,km_driven,fuel,owner,selling_price
3849,Volvo,2000,Diesel,First Owner,2475000
4259,Volvo,20000,Diesel,First Owner,3800000
2472,Mahindra,70000,Diesel,First Owner,1125000
3119,Maruti,110000,Diesel,Fourth & Above Owner,350000
3971,Maruti,20000,Petrol,First Owner,525000


In [133]:
data["brand"].value_counts()

,count
brand,
Maruti,2448
Hyundai,1415
Mahindra,772
Tata,734
Toyota,488
Honda,467
Ford,397
Chevrolet,230
Renault,228


In [134]:
len(data["brand"].unique())

32

In [135]:
# Create a threshold value. For all counts < threshold, categorize into 'Others'
threshold = 100

# get all categories and their numbers
counts = data["brand"].value_counts()

# find out the categories with values less than threshold
categories_to_replace = counts[counts < threshold].index

# replace them into a common category "Others"
data["brand"] = data["brand"].replace(categories_to_replace, "Others")

data["brand"].value_counts()

,count
brand,
Maruti,2448
Hyundai,1415
Mahindra,772
Tata,734
Others,538
Toyota,488
Honda,467
Ford,397
Chevrolet,230


In [147]:
X_train, X_test, y_train, y_test = train_test_split(data.drop("selling_price",axis=1), data["selling_price"], test_size = 0.2, random_state = 23)

In [148]:
X_train.sample(5)

,brand,km_driven,fuel,owner
1393,Skoda,29000,Petrol,Second Owner
5869,Mahindra,120000,Diesel,First Owner
3773,Maruti,8500,Petrol,First Owner
576,Maruti,97500,Petrol,First Owner
180,Volkswagen,13000,Petrol,First Owner


In [149]:
X_train["brand"].nunique()

13

In [150]:
X_train_encoded = pd.get_dummies(X_train, columns=["brand"],drop_first=True, dtype=int)
X_test_encoded = pd.get_dummies(X_test, columns=["brand"],drop_first=True, dtype=int)

In [157]:
# X_train_encoded.sample(5)
X_test_encoded.sample(5)

,km_driven,fuel,owner,brand_Chevrolet,brand_Ford,brand_Honda,brand_Hyundai,brand_Mahindra,brand_Maruti,brand_Others,brand_Renault,brand_Skoda,brand_Tata,brand_Toyota,brand_Volkswagen
107,37800,Petrol,First Owner,0,0,0,1,0,0,0,0,0,0,0,0
2001,54000,Petrol,First Owner,0,0,0,0,0,1,0,0,0,0,0,0
5370,100000,Diesel,Second Owner,0,0,0,0,0,0,0,0,0,0,0,1
6387,100000,Petrol,Third Owner,0,0,0,0,0,1,0,0,0,0,0,0
233,70000,Diesel,First Owner,0,0,0,0,1,0,0,0,0,0,0,0
